# 05 — Mô hình 3: CNN (đóng băng) + GRU  ·  bản TÍCH HỢP (TV3)

Dùng **chung hạ tầng của nhóm** (khác notebook 05 cũ tự làm mọi thứ):
- Dữ liệu/nhãn/chia tập: `src.data.loader` + `src.data.labels` trên `nhom05-falldet-processed`.
- Tracks: `tracks.parquet` chính thức của TV1 (`scripts/make_tracks.py`, Kaggle Dataset `nhom05-falldet-tracks`), dùng chung với M1.
- Đánh giá: `scripts/evaluate.py` trên `predictions.csv` — **cùng độ đo/T/split với M1, M2**.

**Add Input:** Kaggle Dataset `nhom05-falldet-processed`. Bật **GPU** và **Internet**.

## 0. Lấy mã nguồn + cố định seed

In [ ]:
import os, sys
BRANCH = 'main'
REPO = '/kaggle/working/FallDetection'
if not os.path.exists(REPO):
    !git clone --depth 1 -b {BRANCH} https://github.com/ducqwang/FallDetection.git {REPO}
%cd {REPO}
!pip install -q ultralytics==8.4.202 lap==0.5.14
sys.path.insert(0, REPO)
from src.data.env_info import setup_notebook
setup_notebook('configs/data.yaml', seed=42)

In [ ]:
from pathlib import Path
from src.data.loader import data_root
ROOT = str(data_root())
META = f'{ROOT}/metadata.csv'
GT   = f'{ROOT}/annotations/le2i_gt_bbox.csv'
WORK = Path('/kaggle/working/m3'); WORK.mkdir(exist_ok=True)
TRACKS = WORK / 'tracks' / 'tracks.parquet'  # hoặc /kaggle/input/nhom05-falldet-tracks/tracks.parquet
CKPT   = WORK / 'm3_best.pt'
PRED   = WORK / 'm3_pred.csv'
RESULTS = 'reports/results'
print('Dữ liệu:', ROOT)

## 1. Tạo tracks (dùng CHUNG với M1)

`make_tracks.py` sinh `tracks.parquet` chính thức đúng schema nhóm (cùng tham số mặc định với M1). Nếu đã có Kaggle Dataset
`nhom05-falldet-tracks`, **Add Input** nó, trỏ `TRACKS` vào và bỏ qua ô này — khi đó M1/M2/M3 dùng y hệt một tracks.

In [ ]:
!python scripts/make_tracks.py --root {ROOT} --out-dir {TRACKS.parent}

## 2. Huấn luyện M3 (within-dataset: Le2i)

Chọn track chính theo IoU với GT của Le2i; cắt cửa sổ trượt; gán nhãn bằng `window_label` dùng chung.
Chỉ huấn luyện phần GRU (backbone CNN đóng băng).

In [ ]:
!python -m src.models.m3_cnn_gru train \
    --tracks {TRACKS} --meta {META} --root {ROOT} \
    --dataset le2i --gt-bbox {GT} --backbone mobilenet_v2 --epochs 30 --ckpt {CKPT}

## 3. Dự đoán → predictions.csv (schema chung)

Chấm `fall_score` cho **mọi track** của mọi video có trong `tracks.parquet` (cả URFD lẫn Le2i),
để vừa đánh giá within-dataset vừa cross-dataset bằng cùng một file.

In [ ]:
!python -m src.models.m3_cnn_gru predict \
    --tracks {TRACKS} --meta {META} --root {ROOT} --ckpt {CKPT} --out {PRED}

## 4. Đánh giá bằng script chung (cùng M1/M2)

Within-dataset Le2i (test) với T = 0.5, 1.0, 2.0 giây.

In [ ]:
!python scripts/evaluate.py --pred {PRED} --meta {META} \
    --split test --dataset le2i --T 0.5 1.0 2.0 --tag m3

## 5. Cross-dataset (Protocol B: train Le2i → test toàn bộ URFD)

Dùng chính `predictions.csv` ở trên, chỉ đổi tập đánh giá sang URFD (`--split all`).

In [ ]:
!python scripts/evaluate.py --pred {PRED} --meta {META} \
    --split all --dataset urfd --tag m3_cross_B

---
**Ghi chú tích hợp:** M3 không còn tự index dữ liệu / tự chạy YOLO / tự đánh giá.
Mọi thứ đi qua `src.data.*` + `tracks.parquet` + `scripts/evaluate.py` giống M1 (TV1) và M2 (TV2).
Bảng kết quả `reports/results/m3_*` so sánh trực tiếp được với `m1_*`.